# Fine-tune DistilBERT to recognise a lecture's subject

Run on **Kaggle** with **Settings → Accelerator: GPU T4 x2** and **Internet: On**.
Then *Run All* (about 15–25 minutes).

What it does:
1. Builds exactly the same train / validation / test split as `baseline.py` (the `data.py` cell below is a copy).
2. Cuts every lecture into pieces of 512 tokens (DistilBERT's limit) and trains on the pieces.
3. Classifies a whole lecture by averaging the predictions of its pieces.
4. Picks the best epoch on validation, scores the **test** split once, and saves:
   - `results_distilbert.json` – the scores (same format as `results/baseline.json`)
   - `subject-distilbert.zip` – the model, to download and unzip into `subject_classifier/models/distilbert/`

In [ ]:
%%writefile data.py
"""Builds the subject-classification dataset from TIB (gigant/tib).

Every TIB record has a `subject` field, sometimes several subjects separated by
newlines. We keep lectures with exactly ONE subject from the classes below
(each has at least ~50 single-subject lectures), pool TIB's own splits, and make
a fixed, stratified 70 / 15 / 15 train / validation / test split.

80% of TIB is Computer Science, so the TRAINING split caps Computer Science at
CS_TRAIN_CAP examples; validation and test keep the natural mix.

This file is self-contained so the Kaggle notebook can copy it unchanged.
"""
import random
from collections import Counter

CLASSES = [
    "Computer Science",
    "Mathematics",
    "Chemistry",
    "Physics",
    "Information Science",
    "Linguistics",
    "Economics and Social Sciences",
    "Engineering",
]
MAX_WORDS = 3000       # enough text for any model here; keeps memory low
CS_TRAIN_CAP = 1000
SEED = 42


def load_examples():
    """[(text, label)] for single-subject lectures in CLASSES, read row by row."""
    from datasets import load_dataset

    ds = load_dataset("gigant/tib")
    examples = []
    for split in ds:
        for row in ds[split].select_columns(["transcript", "subject"]):
            subject = (row["subject"] or "").strip()
            text = (row["transcript"] or "").split()
            if subject in CLASSES and len(text) >= 100:
                examples.append((" ".join(text[:MAX_WORDS]), subject))
    return examples


def split_examples(examples):
    """Stratified 70/15/15 split with a fixed seed, then cap CS in train only."""
    rng = random.Random(SEED)
    by_label = {}
    for ex in examples:
        by_label.setdefault(ex[1], []).append(ex)

    train, valid, test = [], [], []
    for label in CLASSES:
        items = by_label.get(label, [])
        rng.shuffle(items)
        n_test = round(len(items) * 0.15)
        n_valid = round(len(items) * 0.15)
        test += items[:n_test]
        valid += items[n_test:n_test + n_valid]
        train += items[n_test + n_valid:]

    cs = [ex for ex in train if ex[1] == "Computer Science"]
    rest = [ex for ex in train if ex[1] != "Computer Science"]
    rng.shuffle(cs)
    train = rest + cs[:CS_TRAIN_CAP]

    for part in (train, valid, test):
        rng.shuffle(part)
    return train, valid, test


def load_splits():
    return split_examples(load_examples())


if __name__ == "__main__":
    train, valid, test = load_splits()
    for name, part in (("train", train), ("valid", valid), ("test", test)):
        counts = Counter(label for _, label in part)
        print(f"{name}: {len(part)}  " + ", ".join(f"{k}: {counts[k]}" for k in CLASSES))

In [ ]:
import json, os, shutil, time
import numpy as np
import torch
from torch import nn
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from data import CLASSES, load_splits

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 512
CHUNKS_PER_LECTURE = 4      # pieces per lecture (the first 4 x ~500 tokens)
EPOCHS = 4
LR = 4e-5
set_seed(42)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - turn on the GPU!")

In [ ]:
train, valid, test = load_splits()
print({k: len(v) for k, v in [("train", train), ("valid", valid), ("test", test)]})
label2id = {c: i for i, c in enumerate(CLASSES)}
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_chunks(examples):
    """One row per piece; `lecture` remembers which lecture a piece came from."""
    rows = {"input_ids": [], "attention_mask": [], "labels": [], "lecture": []}
    body = MAX_LEN - 2
    for i, (text, label) in enumerate(examples):
        ids = tok(text, add_special_tokens=False, truncation=False)["input_ids"]
        for start in range(0, min(len(ids), body * CHUNKS_PER_LECTURE), body):
            piece = [tok.cls_token_id] + ids[start:start + body] + [tok.sep_token_id]
            if len(piece) < 64 and start > 0:
                break  # skip a tiny tail
            rows["input_ids"].append(piece)
            rows["attention_mask"].append([1] * len(piece))
            rows["labels"].append(label2id[label])
            rows["lecture"].append(i)
    return Dataset.from_dict(rows)

t = time.time()
ds_train, ds_valid, ds_test = to_chunks(train), to_chunks(valid), to_chunks(test)
print(f"pieces: train {len(ds_train)}, valid {len(ds_valid)}, test {len(ds_test)}  ({time.time()-t:.0f}s)")

In [ ]:
# Computer Science is still half of the training pieces, so weight the loss by class.
counts = np.bincount(ds_train["labels"], minlength=len(CLASSES))
class_weights = torch.tensor(len(ds_train["labels"]) / (len(CLASSES) * np.maximum(counts, 1)), dtype=torch.float)
print(dict(zip(CLASSES, class_weights.round(decimals=2).tolist())))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.functional.cross_entropy(outputs.logits, labels, weight=class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

def lecture_level(pred_logits, dataset, examples):
    """Average the softmax of each lecture's pieces -> one prediction per lecture."""
    probs = torch.softmax(torch.tensor(pred_logits, dtype=torch.float), dim=-1).numpy()
    lec = np.array(dataset["lecture"])
    summed = np.zeros((len(examples), len(CLASSES)))
    np.add.at(summed, lec, probs)
    y_pred = [CLASSES[i] for i in summed.argmax(1)]
    y_true = [label for _, label in examples]
    return y_true, y_pred

def scores(y_true, y_pred):
    return {
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "macro_f1": round(f1_score(y_true, y_pred, labels=CLASSES, average="macro", zero_division=0), 4),
        "per_class_f1": dict(zip(CLASSES, [round(x, 4) for x in f1_score(
            y_true, y_pred, labels=CLASSES, average=None, zero_division=0)])),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=CLASSES).tolist(),
    }

def compute_metrics(eval_pred):
    y_true, y_pred = lecture_level(eval_pred.predictions, ds_valid, valid)
    return {"lecture_macro_f1": f1_score(y_true, y_pred, labels=CLASSES, average="macro", zero_division=0),
            "lecture_accuracy": accuracy_score(y_true, y_pred)}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(CLASSES), id2label=dict(enumerate(CLASSES)), label2id=label2id)

args = TrainingArguments(
    output_dir="out", num_train_epochs=EPOCHS, learning_rate=LR, weight_decay=0.01,
    # 10% warm-up, given as steps: newer transformers removed warmup_ratio
    warmup_steps=int(0.1 * EPOCHS * len(ds_train) / (16 * max(1, torch.cuda.device_count()))),
    per_device_train_batch_size=16, per_device_eval_batch_size=64, fp16=torch.cuda.is_available(),
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="lecture_macro_f1", greater_is_better=True,
    logging_steps=50, report_to="none", seed=42,
)
trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=ds_train.remove_columns(["lecture"]),
    eval_dataset=ds_valid.remove_columns(["lecture"]),
    data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics,
)
t = time.time()
trainer.train()
print(f"training took {(time.time()-t)/60:.1f} min")

In [ ]:
# Score the TEST split once, with the epoch that was best on validation.
pred = trainer.predict(ds_test.remove_columns(["lecture"])).predictions
y_true, y_pred = lecture_level(pred, ds_test, test)
print(classification_report(y_true, y_pred, labels=CLASSES, zero_division=0, digits=3))

best_valid = max(r["eval_lecture_macro_f1"] for r in trainer.state.log_history if "eval_lecture_macro_f1" in r)
results = {"model": MODEL_NAME, "max_len": MAX_LEN, "chunks_per_lecture": CHUNKS_PER_LECTURE,
           "epochs": EPOCHS, "learning_rate": LR, "valid_macro_f1": round(best_valid, 4),
           "test": scores(y_true, y_pred)}
json.dump(results, open("results_distilbert.json", "w"), indent=2)
print("TEST  acc", results["test"]["accuracy"], " macro-F1", results["test"]["macro_f1"])

In [ ]:
# Save the model (+ tokenizer and settings) and zip it for download from the Output tab.
out = "subject-distilbert"
trainer.save_model(out)
tok.save_pretrained(out)
json.dump({"max_len": MAX_LEN, "chunks_per_lecture": CHUNKS_PER_LECTURE}, open(f"{out}/inference.json", "w"))
shutil.make_archive("subject-distilbert", "zip", out)
shutil.rmtree("out", ignore_errors=True)  # checkpoints are not needed
print(os.listdir("."))